# Etapa 06 — ¿Qué hace viable una respuesta y quién asume sus efectos?

Consulta el dossier. Predice, calcula, explica y juzga el alcance antes de registrar tu decisión.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
# Encuentra la raíz tanto desde la carpeta del notebook como desde la raíz del libro.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'sa_mise' / '__init__.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from sa_mise import datos, modelos, expediente
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': .2})

In [2]:
from sa_mise import sector
CASO='ejemplo_docente'
# Usa el mismo identificador de tu equipo en las ocho etapas.

## 1. Una fecha condicionada cambia disponibilidad
Para aislar la demora, usaremos una sola trayectoria con el primer año sin solar adicional y el segundo con ella. La secuencia de aportes, el calendario y el stock se conservan. Antes de calcular, identificamos la condición que permitiría acreditar una fecha.

In [3]:
# Usamos la trayectoria completa con y sin solar, y una implementación mensual
# explícita para conservar calendario y estado cuando cambia la capacidad.
from inspect import signature
print('Variables controlables del laboratorio:',signature(sector.sistema_docente))
condiciones=pd.DataFrame([
 ['Entrada solar','Fecha de conexión documentada','Pendiente','Sin aporte adicional hasta verificación'],
 ['Reducción demanda','Mecanismo y participantes identificados','Pendiente','Reducción no acreditada'],
 ['Reserva agua','Factibilidad y reglas de operación','Pendiente','No dar orden de despacho']
],columns=['alternativa','evidencia','estado','consecuencia'])
display(condiciones)

Variables controlables del laboratorio: (aportes=None, solar_extra_MW=0.0, termica_disponible=0.9, reduccion_demanda=0.0, reserva_MWh=0.0, stock_inicial=120000.0)


,alternativa,evidencia,estado,consecuencia
0,Entrada solar,Fecha de conexión documentada,Pendiente,Sin aporte adicional hasta verificación
1,Reducción demanda,Mecanismo y participantes identificados,Pendiente,Reducción no acreditada
2,Reserva agua,Factibilidad y reglas de operación,Pendiente,No dar orden de despacho


La matriz evita dar por disponible una intervención porque exista en el menú del modelo. Ahora calcularemos la demora con una serie mensual de capacidad adicional, sin cambiar aportes ni estado inicial.

## 2. Costo físico de una demora supuesta
La variable solar adicional acepta una serie de 24 meses. El cambio ocurre en el mes trece. El experimento no identifica la causa ni la probabilidad de un atraso real.

In [4]:
aportes=sector.aportes_docentes()*.45
a_tiempo=sector.sistema_docente(aportes,solar_extra_MW=20.)
tardia=sector.sistema_docente(aportes,solar_extra_MW=np.r_[np.zeros(12),np.full(12,20.)])
display(pd.DataFrame({'caso':['A tiempo','Mes 13'],'faltante_MWh':[a_tiempo.faltante_MWh.sum(),tardia.faltante_MWh.sum()],
 'termica_MWh':[a_tiempo.termica_MWh.sum(),tardia.termica_MWh.sum()],'stock_final_MWh_eq':[a_tiempo.stock_final_MWh_eq.iloc[-1],tardia.stock_final_MWh_eq.iloc[-1]]}))

,caso,faltante_MWh,termica_MWh,stock_final_MWh_eq
0,A tiempo,257932.475019,1.281668e+06,0.0
1,Mes 13,273269.975019,1.301466e+06,0.0


Compara faltante y uso térmico: una demora puede afectar una métrica sin afectar otra. Si no cambia el faltante, no concluyas que la fecha es irrelevante; observa sustitución y stock. La condición institucional debe relacionarse con la decisión sin atribuir motivos a actores reales.

## Registrar el avance del equipo
Sustituye el campo de decisión por tu razonamiento y conserva el nivel de evidencia. El identificador es el mismo durante todo el caso.

In [5]:
decision_estudiante='POR COMPLETAR'
expediente.registrar(CASO,'gobernanza','Demora solar: faltante a tiempo='+str(a_tiempo.faltante_MWh.sum())+'; tardío='+str(tardia.faltante_MWh.sum()),'descripcion' if 6 in [0,1] else 'exploracion',
 'Extractos locales y/o supuestos identificados en DOSSIER y esta etapa',
 'Transformaciones, calendario y unidades explícitos en las celdas precedentes',
 'No constituye evaluación completa del SIN; distinguir cada resultado observado de los sintéticos',
 decision_estudiante)

Out[5]: 
{'version': 2,
 'caso': 'ejemplo_docente',
 'etapas': {'evidencia': {'hallazgo': 'Año 2024: generación 83262.923 GWh y pico 11704.365 MW; son magnitudes diferentes',
   'nivel': 'descripcion',
   'fuente': 'Extractos locales y/o supuestos identificados en DOSSIER y esta etapa',
   'transformacion': 'Transformaciones, calendario y unidades explícitos en las celdas precedentes',
   'limite': 'No constituye evaluación completa del SIN; distinguir cada resultado observado de los sintéticos',
   'decision_estudiante': 'POR COMPLETAR'},
  'identidad': {'hallazgo': "Asociaciones mensuales: {'meses': 264, 'correlacion_bruta': -0.3654022452929349, 'correlacion_residual': -0.5459860648178948, 'interpretacion': 'Asociación; controles aditivos no identifican causalidad ni corrigen toda no estacionariedad'}",
   'nivel': 'descripcion',
   'fuente': 'Extractos locales y/o supuestos identificados en DOSSIER y esta etapa',
   'transformacion': 'Transformaciones, calendario y unidades explícit